## Overview

This notebook displays the mapping of meters to their associated high-voltage transformers and substations based on GIS connectivity. Users can input a specific meter for mapping or view the entire grid.

---

## Setup

In [1]:
import getpass
import urllib.parse
import pandas as pd
import plotly.express as px

pd.set_option('display.max_rows', None)

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

%load_ext sql
%sql postgresql://$edm_name:$edm_password@$edm_address/edm
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

# Delete the credential variables for security purposes.
del edm_name, edm_password

EDM server address: ········

EDM login information
Username: ········
Password: ········


---

###  Meter to Substation Mapping

#### Input Parameters
Enter the grid ID. 

In [3]:
# User input for the grid.
grid_id = input('Enter grid ID: ') # e.g. North Central Zone

Enter grid ID: North Central Zone


Get the high-voltage transformer and substation upstream of a specific meter in the grid:

In [4]:
# User input for the meter.
meter_id = input('Enter meter ID: ') # e.g. 6D4Y23

Enter meter ID: 6D4Y23


In [5]:
# Get the high-voltage transformers and substations upstream of a specific meter in the grid. 
substation_query = """
    SELECT grid_element_id as hv_transformer, 
        meta->> 'enclosure_id' as substation_id
    FROM grid_get_sources('{}', '{}', 'true') 
    WHERE grid_id = '{}'
        AND type = 'Transformer'
        AND meta->> 'voltage_level' = 'HV/MV' ; 
    """
formatted_query = substation_query.format(grid_id, meter_id, grid_id)
substation = %sql $formatted_query

# Convert the results to a data frame and display it. 
df_substation = substation.DataFrame()
df_substation['meter_id'] = meter_id

df_substation

,hv_transformer,substation_id,meter_id
0,20593_hvmv,52603,6D4Y23


Get the high-voltage transformer and substations associated with every meter in the grid:

In [6]:
substations_query = """
    SELECT ge.grid_element_id as hv_transformer_id,
        ggd.grid_element_id as meter_id, 
        ge.meta->> 'enclosure_id' as substation_id
    FROM grid_element ge
    JOIN grid_get_downstream('{}', ge.grid_element_id) ggd
        ON True
    WHERE ge.grid_id = '{}'
        AND ge.type = 'Transformer'
        AND ggd.type = 'Meter'
        AND ge.meta->> 'voltage_level' = 'HV/MV'; 
    """
formatted_query = substations_query.format(grid_id, grid_id)
substations = %sql $formatted_query

# Convert the results to a data frame and display it. 
df_substations = substations.DataFrame().set_index(['substation_id', 'hv_transformer_id', 'meter_id'])

df_substations.sort_values(by=['substation_id', 'hv_transformer_id'])

Empty DataFrame
Columns: []
Index: [(17671, 116651_hvmv, 22AY1), (17671, 116651_hvmv, 22AY11), (17671, 116651_hvmv, 22AY13-1), (17671, 116651_hvmv, 22AY13-1ZA), (17671, 116651_hvmv, 22AY17-4), (17671, 116651_hvmv, 22AY17-4ZA), (17671, 116651_hvmv, 22AY1X-4), (17671, 116651_hvmv, 22AY22ZA), (17671, 116651_hvmv, 22AY24), (17671, 116651_hvmv, 22AY24X), (17671, 116651_hvmv, 22AY26-1-1), (17671, 116651_hvmv, 22AY26-3-12), (17671, 116651_hvmv, 22AY26-3-8), (17671, 116651_hvmv, 22AY26-3-8ZA), (17671, 116651_hvmv, 22AY26-3ZA), (17671, 116651_hvmv, 22AY26-4-1), (17671, 116651_hvmv, 22AY26-5), (17671, 116651_hvmv, 22AY26-5ZA), (17671, 116651_hvmv, 22AY26-5ZB), (17671, 116651_hvmv, 22AY26-6), (17671, 116651_hvmv, 22AY27), (17671, 116651_hvmv, 22AY27ZA), (17671, 116651_hvmv, 22AY3), (17671, 116651_hvmv, 22AY30-1), (17671, 116651_hvmv, 22AY30-13-1), (17671, 116651_hvmv, 22AY30-15ZC), (17671, 116651_hvmv, 22AY30-2), (17671, 116651_hvmv, 22AY30-26-1), (17671, 116651_hvmv, 22AY30-29), (17671, 116651_hvmv, 22AY30-29ZE), (17671, 116651_hvmv, 22AY30-2ZA), (17671, 116651_hvmv, 22AY30-3), (17671, 116651_hvmv, 22AY30-33X), (17671, 116651_hvmv, 22AY30-35), (17671, 116651_hvmv, 22AY30-37), (17671, 116651_hvmv, 22AY30-38-1), (17671, 116651_hvmv, 22AY30-39-1), (17671, 116651_hvmv, 22AY30-39-1ZA), (17671, 116651_hvmv, 22AY30-41), (17671, 116651_hvmv, 22AY30-5), (17671, 116651_hvmv, 22AY30-5ZA), (17671, 116651_hvmv, 22AY30-7), (17671, 116651_hvmv, 22AY30-A), (17671, 116651_hvmv, 22AY30-AZB), (17671, 116651_hvmv, 22AY30ZA), (17671, 116651_hvmv, 22AY36-1), (17671, 116651_hvmv, 22AY38-1), (17671, 116651_hvmv, 22AY38-2), (17671, 116651_hvmv, 22AY3X1-1), (17671, 116651_hvmv, 22AY3ZA), (17671, 116651_hvmv, 22AY43-5), (17671, 116651_hvmv, 22AY45), (17671, 116651_hvmv, 22AY46), (17671, 116651_hvmv, 22AY49), (17671, 116651_hvmv, 22AY53), (17671, 116651_hvmv, 22AY53ZA), (17671, 116651_hvmv, 22AY53ZC), (17671, 116651_hvmv, 22AY6), (17671, 116651_hvmv, 22AY8-2), (17671, 116651_hvmv, 22BY10-A), (17671, 116651_hvmv, 22BY10-B), (17671, 116651_hvmv, 22BY16), (17671, 116651_hvmv, 22BY25), (17671, 116651_hvmv, 22BY28), (17671, 116651_hvmv, 22BY28X), (17671, 116651_hvmv, 22BY30), (17671, 116651_hvmv, 22BY33X), (17671, 116651_hvmv, 22BY35), (17671, 116651_hvmv, 22BY35-1), (17671, 116651_hvmv, 22BY36-1), (17671, 116651_hvmv, 22BY37A), (17671, 116651_hvmv, 22BY39), (17671, 116651_hvmv, 22BY44), (17671, 116651_hvmv, 22BY46), (17671, 116651_hvmv, 22BY48), (17671, 116651_hvmv, 22BY48ZA), (17671, 116651_hvmv, 22BY48ZB), (17671, 116651_hvmv, 22BY49-1), (17671, 116651_hvmv, 22BY49-3), (17671, 116651_hvmv, 22BY51), (17671, 116651_hvmv, 22BY55), (17671, 116651_hvmv, 22BY8), (17671, 116651_hvmv, 22D1Y14-1), (17671, 116651_hvmv, 22D1Y14-3), (17671, 116651_hvmv, 22D1Y14-3ZA), (17671, 116651_hvmv, 22D1Y15), (17671, 116651_hvmv, 22D1Y17), (17671, 116651_hvmv, 22D1Y17-5), (17671, 116651_hvmv, 22D1Y18), (17671, 116651_hvmv, 22D1Y19), (17671, 116651_hvmv, 22D1Y2), (17671, 116651_hvmv, 22D1Y21-1), (17671, 116651_hvmv, 22D1Y22), (17671, 116651_hvmv, 22D1Y24), (17671, 116651_hvmv, 22D1Y32-1), (17671, 116651_hvmv, 22D1Y41), (17671, 116651_hvmv, 22D1Y41ZA), (17671, 116651_hvmv, 22D1Y41ZB), (17671, 116651_hvmv, 22D1Y41ZD), (17671, 116651_hvmv, 22D2Y12), ...]

In [7]:
# Count the number of meters per transformer. 
meters_per_transformers = df_substations.reset_index().groupby(by=['substation_id', 'hv_transformer_id']).count().rename(
    columns={'meter_id': 'number_of_meters_per_transformer'})
meters_per_transformers

number_of_meters_per_transformer
substation_id hv_transformer_id                                  
17671         116651_hvmv                                     340
              21298_hvmv                                      197
17673         10680_hvmv                                      584
              39660_hvmv                                       66
              41449_hvmv                                      582
23327         35978_hvmv                                      316
              41323_hvmv                                      634
              49306_hvmv                                      204
23969         28667_hvmv                                      569
              40771_hvmv                                      101
52523         10954_hvmv                                      557
              1798_hvmv                                       788
              40135_hvmv                                      354
52537         19612_hvmv                                      731
              32644_hvmv                                      633
52545         121175_hvmv                                     143
              23793_hvmv                                       27
              36412_hvmv                                      785
              42093_hvmv                                      206
52559         103663_hvmv                                     441
              12465_hvmv                                      462
              48504_hvmv                                      162
52561         17128_hvmv                                      116
              37609_hvmv                                      290
              40543_hvmv                                      568
52568         25941_hvmv                                      197
              41583_hvmv                                      274
52575         39159_hvmv                                      171
52584         114629_hvmv                                     352
              45212_hvmv                                      730
              48803_hvmv                                      575
52588         10407_hvmv                                      819
              12373_hvmv                                      316
              28538_hvmv                                      567
              7676_hvmv                                       926
52592         22941_hvmv                                      403
52596         24966_hvmv                                      383
              28748_hvmv                                      622
              48939_hvmv                                      251
52603         20593_hvmv                                     1037
              29610_hvmv                                      199
              36896_hvmv                                      222
52752         21499_hvmv                                      287
52756         10072_hvmv                                      406
              12966_hvmv                                      585

In [8]:
# Count the number of transformers and meters per substation.
substations = meters_per_transformers.reset_index().groupby(by=['substation_id']).count().reset_index().rename(
    columns={'hv_transformer_id': 'number_of_transformers_per_substation'})
substations['number_of_meters_per_substation'] = meters_per_transformers.reset_index().groupby(by=['substation_id']).sum().reset_index()['number_of_meters_per_transformer']
substations.drop(columns=['number_of_meters_per_transformer'],inplace=True)
substations

,substation_id,number_of_transformers_per_substation,number_of_meters_per_substation
0,17671,2,537
1,17673,3,1232
2,23327,3,1154
3,23969,2,670
4,52523,3,1699
5,52537,2,1364
6,52545,4,1161
7,52559,3,1065
8,52561,3,974
9,52568,2,471


---